# Web APIs and the Event Loop

JavaScript itself is strictly single-threaded, but it achieves non-blocking, asynchronous behavior by offloading time-consuming tasks to browser-provided **Web APIs**.

While the JavaScript engine (like Chrome's V8) has only one call stack and can only execute one line of code at a time, the surrounding browser environment is highly multithreaded. Web APIs act as a bridge, allowing JavaScript to delegate heavy operations — such as network requests, timers, and event handling — to background threads managed by the browser.

## 🧱 The Architectural Pieces

To understand how JavaScript interacts with Web APIs without freezing the user interface, you need to look at the complete runtime ecosystem:

- **The Call Stack** — Tracks the current function being executed. Since it is single-threaded, functions must run and finish one by one.
- **Web APIs** — Built-in features provided by the browser environment (not the JavaScript language itself). They handle operations in the background, independently of the main stack.
- **The Callback Queue (Task Queue)** — A holding area where background tasks go once the Web API finishes processing them, waiting to be executed.
- **The Event Loop** — A continuous monitoring mechanism. It waits until the Call Stack is completely empty, then pulls the first task from the Callback Queue and pushes it onto the stack for execution.

## 🔄 Step-by-Step Execution Lifecycle

When you trigger an asynchronous function, the ecosystem coordinates the task through these phases:

```
[ Call Stack ]  --->  (Offloads task)  --->  [ Web APIs (Browser Background) ]
      ^                                                     |
      | (Pushed when stack is empty)                        v (Task finishes)
[ Event Loop ]  <---  (Pulls from queue) <---  [ Callback Queue ]
```

1. **Invocation** — JavaScript reads an asynchronous command (e.g. `fetch()` or `setTimeout`).
2. **Hand-off** — The engine pushes the function to the Call Stack, recognizes it as a Web API, delegates it to the browser, and immediately pops it off the stack.
3. **Background Processing** — The main thread continues running subsequent code seamlessly. Meanwhile, the browser handles the network request or timer in a separate background thread.
4. **Queueing** — When the background operation finishes, the browser places the corresponding callback function into the Callback Queue.
5. **Execution** — The Event Loop verifies that the Call Stack is clear, grabs the callback function, and returns it to the main thread to be executed.

## 🛠️ Common Types of Web APIs

The components handling background processing are categorized by their specific platform functions:

- **Timers** — APIs like `setTimeout` and `setInterval` calculate timing sequences without blocking the main stack.
- **Network Requests** — The [Fetch API](https://developer.mozilla.org/en-US/docs/Web/API/Fetch_API) and `XMLHttpRequest` manage HTTP data transfers entirely on background browser threads.
- **DOM Events** — Listeners like `element.addEventListener` wait for user interactions (clicks, scrolls) using browser event-detection systems.
- **Web Workers** — The [Web Workers API](https://developer.mozilla.org/en-US/docs/Web/API/Web_Workers_API) lets you explicitly spawn completely separate OS-level threads to compute complex data without freezing the UI.